# Task 2

### Create a 3D-graph with "center of molecule" as the node and distance to its neighbouring molecules as the edges

In [75]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import miniball
import networkx as nx
from numba import jit

## Get data from npt-HK4.gro file

In [76]:
"""
Reads the npt-HK4.gro and returns the following:

data(pd.Dataframe): A dataframe that contains Residue ID(res_id) and Name(res_name), Atom Name (atom_name) and ID(atom_id), the atom coordinates(x, y, z) and velocity components(Vx, Vy, Vz)
title(str): The title of the .gro file
num_atoms(str): The number of atoms
box_dimensions(list): The simulation box dimensions in a list of [x,y,z] 

"""
# setting column specifications
colspecs = [
    (0, 5),
    (5, 8),
    (8, 15),
    (15, 20),
    (20, 28),
    (28, 36),
    (36, 44),
    (44, 52),
    (52, 60),
    (60, 68),
]

# setting names of columns
names = ["res_id", "res_name", "atom_name", "atom_id", "x", "y", "z", "Vx", "Vy", "Vz"]

# reading data
data = pd.read_fwf(
    "../../../data/npt-HK4.gro",
    colspecs=colspecs,
    names=names,
    skiprows=2,
    skipfooter=1,
)

# Problem: after atom id 99999, it reverts back to 0. The following solves this:
for i in data.index:
    if i >= 99999:
        data.at[i, "atom_id"] += 100000

# reading Title, number of atoms and box_dimensions
with open("../../../data/npt-HK4.gro", "rb") as f:
    title = f.readline().decode().strip()  # First line of .gro file
    num_atoms = f.readline().decode().strip()  # Second line of .gro file

    # Last line of file
    f.seek(-2, 2)
    while f.read(1) != b"\n":
        f.seek(-2, 1)
    box_dimensions = f.readline().decode().strip().split()
    box_dimensions = list(map(float, box_dimensions))


print(data.tail())
print(f"Title: {title}")
print(f"Number of atoms: {num_atoms}")
print(f"Box dimensions: {box_dimensions}")

        res_id res_name atom_name  atom_id  ...      z      Vx      Vy      Vz
126079    1501      HK4       H23   126080  ...  1.494 -1.2440  0.1068 -0.7226
126080    1501      HK4       C44   126081  ...  1.525  0.1513  0.5200  0.5632
126081    1501      HK4       H24   126082  ...  1.425  0.3100  2.4172  1.1123
126082    1501      HK4       C45   126083  ...  1.615 -0.0110 -0.1637  0.2235
126083    1501      HK4       H25   126084  ...  1.607 -0.7669 -0.9514  2.2265

[5 rows x 10 columns]
Title: mol only system in water
Number of atoms: 126084
Box dimensions: [11.24798, 11.24798, 11.24798]


In [77]:
# Dropping velocity components and setting index
data.drop(columns=["Vx", "Vy", "Vz"], inplace=True)  
data.set_index(['res_id', 'atom_name'], inplace=True) 

data

res_name  atom_id      x      y      z
res_id atom_name                                       
1      H28            HK4        1  1.602  0.962  0.912
       C48            HK4        2  1.565  0.879  0.852
       C47            HK4        3  1.655  0.809  0.773
       H27            HK4        4  1.754  0.855  0.756
       C46            HK4        5  1.603  0.717  0.683
...                   ...      ...    ...    ...    ...
1501   H23            HK4   126080  0.953  5.984  1.494
       C44            HK4   126081  0.736  6.009  1.525
       H24            HK4   126082  0.707  6.041  1.425
       C45            HK4   126083  0.632  5.988  1.615
       H25            HK4   126084  0.542  6.051  1.607

[126084 rows x 5 columns]

## Midpoint finding algorithm to find the "center of the molecule", defined as the midpoint of both oxygen atoms in the molecule

In [78]:
def compute_midpoints_df(filtered_data, box_length):
    """
    Given a multi-indexed DataFrame (filtered_data) and box_length,
    returns a DataFrame with columns ['res_id', 'mid_x', 'mid_y', 'mid_z']
    containing the midpoints between O1 and O2 atoms for each residue.
    """
    def minimum_image(dx, box_length):
        reciprocal_half_box = 1.0 / (0.5 * box_length)
        k = int(dx * reciprocal_half_box)
        return dx - k * box_length

    def midpoint_pbc(p1, p2, box_length):
        dx = p2 - p1
        dx_mic = minimum_image(dx, box_length)
        midpoint = (p1 + 0.5 * dx_mic) % box_length
        return midpoint

    midpoints = []
    for res_id in filtered_data.index.get_level_values(0).unique():
        try:
            o1 = filtered_data.loc[(res_id, "O1")]
            o2 = filtered_data.loc[(res_id, "O2")]
            mx = midpoint_pbc(o1['x'], o2['x'], box_length)
            my = midpoint_pbc(o1['y'], o2['y'], box_length)
            mz = midpoint_pbc(o1['z'], o2['z'], box_length)
            midpoints.append({'res_id': res_id, 'mid_x': mx, 'mid_y': my, 'mid_z': mz})
        except KeyError:
            # Skip residues without both O1 and O2
            continue

    return pd.DataFrame(midpoints)

In [79]:
# Extract box length from box_dimensions string 
box_length = box_dimensions[0]

midpoints_df = compute_midpoints_df(data, box_length)

midpoints_df

,res_id,mid_x,mid_y,mid_z
0,1,1.55850,0.40900,11.16949
1,2,0.44150,0.65700,10.78750
2,3,0.76450,10.72000,0.89800
3,4,0.15751,0.97750,0.13500
4,5,5.54400,0.11051,5.77600
...,...,...,...,...
1496,1497,8.55200,6.78500,8.62900
1497,1498,8.60450,0.93600,7.08450
1498,1499,5.25950,0.52650,11.24649
1499,1500,5.92950,10.45100,3.18150


In [80]:
# Add midpoint coordinates as new columns to filtered_data at the residue level
for idx, row in midpoints_df.iterrows():
    data.loc[(row['res_id'], slice(None)), 'midO_x'] = row['mid_x']
    data.loc[(row['res_id'], slice(None)), 'midO_y'] = row['mid_y']
    data.loc[(row['res_id'], slice(None)), 'midO_z'] = row['mid_z']

data

res_name  atom_id      x  ...  midO_x  midO_y    midO_z
res_id atom_name                           ...                          
1      H28            HK4        1  1.602  ...  1.5585   0.409  11.16949
       C48            HK4        2  1.565  ...  1.5585   0.409  11.16949
       C47            HK4        3  1.655  ...  1.5585   0.409  11.16949
       H27            HK4        4  1.754  ...  1.5585   0.409  11.16949
       C46            HK4        5  1.603  ...  1.5585   0.409  11.16949
...                   ...      ...    ...  ...     ...     ...       ...
1501   H23            HK4   126080  0.953  ...  0.2115   5.783   1.27700
       C44            HK4   126081  0.736  ...  0.2115   5.783   1.27700
       H24            HK4   126082  0.707  ...  0.2115   5.783   1.27700
       C45            HK4   126083  0.632  ...  0.2115   5.783   1.27700
       H25            HK4   126084  0.542  ...  0.2115   5.783   1.27700

[126084 rows x 8 columns]

## Made subboxes. To define which molecules are "near" each other.
Molecules are near each other if:
- They exist within the same subbox
- They are in adjacent subboxes

Each subbox should contain 1 molecule on average.

In [81]:
## First, find dimension of one molecule to approximate number of subboxes needed
molecule_size = pd.DataFrame(columns=["res_id", "molecule_size"])
for n in range(1, 1502):
    one_molecule = data.loc[n, ["x", "y", "z"]]
    molecule_dimensions = (one_molecule.max() - one_molecule.min()).to_list()
    molecule_size.loc[n - 1] = [n, np.prod(molecule_dimensions)]

### My first attempt, but this is number of subboxes per axis, could be useful
# number_of_subboxes = [round(x / y) for x,y in zip(box_dimensions,molecule_dimensions)]
# number_of_subboxes

number_of_subboxes = round(np.prod(box_dimensions) / np.prod(molecule_dimensions))
## Output: number_of_subboxes = 142

## Subbox dimension = box_dimension/mol_dimension + some arbitrary constant to make it slightly bigger to lower chance of edge case where no molecule exist in a subbox
## Use the closest cube root(ie. 125)to provide a better approximation as well as to make calculations simpler

####LOGIC ERROR: WE USE 1501 SUBBOXES, BUT SINCE 1501 IS NOT A PERFECT CUBE, USE 1331 SINCE NEAREST PERFECT CUBE

In [82]:
"""
We self define the number of subboxes needed as 1331 as the total number of molecules is 1501 and we take the closest cube root to it.

Returns:
number_of_subboxes(int): number of subboxes needed
subboxes_dimensions(list): A list of dimensions of a subbox
"""


number_of_subboxes = 1331  #Define number of subboxes needed
subboxes_dimensions = [x / number_of_subboxes ** (1 / 3) for x in box_dimensions] #get subbox dimensions and put into a list

In [83]:
"""
With the dimensions, we create a dataframe of each subbox with the nodes(defined as center of molecule) in the subbox

Returns:
subbox(pd.Dataframe): A dataframe that contains the Index of the Subbox as a Tuple(index), Residue ID(res_ID), the Center of Molecule coordinates(x, y, z)

"""



# Make a list of all possible indices of subboxes, additionally with an outer layer
matrix = [] #Empty list to append into later
n = round(number_of_subboxes ** (1 / 3)) # Maximum indices for the box, excluding outer layer
# Nested for loop to create combinations of 3 from a set of numbers
for i in range(1, n + 1):
    for j in range(1, n + 1):
        for k in range(1, n + 1):
            matrix.append([i, j, k])

# Find molecules within a subbox and insert it as a new row into a dataframe
subbox = pd.DataFrame(columns=["index", "res_id", "x", "y", "z"])
iterate = 1
for p, q, r in matrix:
    molecule_in_subbox = midpoints_df.loc[
        (midpoints_df["mid_x"] < subboxes_dimensions[0] * p)
        & (midpoints_df["mid_y"] < subboxes_dimensions[1] * q)
        & (midpoints_df["mid_z"] < subboxes_dimensions[2] * r)
        & (midpoints_df["mid_x"] >= subboxes_dimensions[0] * (p - 1))
        & (midpoints_df["mid_y"] >= subboxes_dimensions[1] * (q - 1))
        & (midpoints_df["mid_z"] >= subboxes_dimensions[2] * (r - 1))
    ]

    subbox.loc[iterate] = {
        "index": (p, q, r),
        "res_id": molecule_in_subbox["res_id"].values,
        "x": molecule_in_subbox["mid_x"].values,
        "y": molecule_in_subbox["mid_y"].values,
        "z": molecule_in_subbox["mid_z"].values,
    }
    iterate += 1

subbox

,index,res_id,x,y,z
1,"(1, 1, 1)",[4],[0.15751000000000037],[0.9774999999999999],[0.135]
2,"(1, 1, 2)",[],[],[],[]
3,"(1, 1, 3)",[979],[0.764],[0.7645],[2.083]
4,"(1, 1, 4)","[332, 977, 1171]","[0.12500999999999962, 0.18, 0.5825]","[0.004510000000001568, 0.24600000000000002, 0....","[3.2005, 3.944, 3.33]"
5,"(1, 1, 5)",[343],[0.199],[0.5265],[5.0825]
...,...,...,...,...,...
1327,"(11, 11, 7)",[],[],[],[]
1328,"(11, 11, 8)",[1350],[10.836500000000001],[10.552],[8.15]
1329,"(11, 11, 9)",[1442],[10.852],[10.82],[8.9085]
1330,"(11, 11, 10)",[1013],[10.664],[10.6395],[9.4485]


In [84]:
"""
Make a new multiindexed dataframe that contains the subbox with their respective adjacent subboxes and the Molecules within each adjacent subbox

Returns:
edges_df(pd.Dataframe): A Multiindex Dataframe that contains the subboxes(select_index), its respective adjacent subboxes(adjacent_index), and the nodes within the subboxes(res_id) 

"""
# Create empty lists to append to
edges_res_id = []
edges_index = []

# Select a box
for n in subbox.index:
    selected_box = subbox.at[n,"index"] 
    selected_box_str = ",".join(map(str,selected_box))
# Get indices of adjacent boxes to the selected box
    adjacent_boxes = []
    for (p,q,r) in [list(selected_box)]:
        for i in [-1,0,1]:
            for j in [-1,0,1]:
                for k in [-1,0,1]:
                    x = p + i
                    y = q + j
                    z = r + k
                    match x:
                        case 0:
                            x = 11
                        case 12:
                            x = 1
                        case _:
                            x = x
                    match y:
                        case 0:
                            y = 11
                        case 12:
                            y = 1
                        case _:
                            y = y
                    match z:
                        case 0:
                            z = 11
                        case 12:
                            z = 1
                        case _:
                            z = z
                    adjacent_boxes.append((x,y,z))
        # adjacent_boxes.remove((p,q,r)) # Uncomment this line to exclude itself from adjacent_boxes
# For loop iterating over each adjacent box to find the nodes inside them, then append into list
    for (p,q,r) in adjacent_boxes:
        adjacent_box_str = ",".join(map(str,(p,q,r)))
        adjacent_nodes = subbox.loc[subbox["index"] == (p, q, r), "res_id"]
        if adjacent_nodes.empty:
                    continue
        edges_res_id.append(adjacent_nodes.values[0]) 
        edges_index.append((selected_box_str,adjacent_box_str))

edges_df = pd.DataFrame(data = {'res_id': edges_res_id}, index = pd.MultiIndex.from_tuples(edges_index, names=['select_index', 'adjacent_index']))
edges_df.head(28)

res_id
select_index adjacent_index               
1,1,1        11,11,11               [1181]
             11,11,1                    []
             11,11,2                 [147]
             11,1,11            [127, 780]
             11,1,1                  [704]
             11,1,2                 [1366]
             11,2,11            [274, 660]
             11,2,1            [226, 1031]
             11,2,2            [897, 1202]
             1,11,11             [18, 850]
             1,11,1                    [3]
             1,11,2                 [1380]
             1,1,11                    [2]
             1,1,1                     [4]
             1,1,2                      []
             1,2,11            [984, 1343]
             1,2,1            [1206, 1330]
             1,2,2                      []
             2,11,11                 [943]
             2,11,1           [1088, 1308]
             2,11,2                     []
             2,1,11          [1, 164, 244]
             2,1,1                   [847]
             2,1,2                      []
             2,2,11                 [1055]
             2,2,1                   [359]
             2,2,2                   [247]
1,1,2        11,11,1                    []

In [ ]:
#CHANGES: ADDED MODULES
from optimized_subbox_division import subbox_division, flatten_subboxes
edges_df = subbox_division(number_of_subboxes, box_dimensions, midpoints_df)
edges_df.head(28)

res_id
select_index adjacent_index               
1,1,1        11,11,11               [1181]
             11,11,1                    []
             11,11,2                 [147]
             11,1,11            [127, 780]
             11,1,1                  [704]
             11,1,2                 [1366]
             11,2,11            [274, 660]
             11,2,1            [226, 1031]
             11,2,2            [897, 1202]
             1,11,11             [18, 850]
             1,11,1                    [3]
             1,11,2                 [1380]
             1,1,11                    [2]
             1,1,1                     [4]
             1,1,2                      []
             1,2,11            [984, 1343]
             1,2,1            [1206, 1330]
             1,2,2                      []
             2,11,11                 [943]
             2,11,1           [1088, 1308]
             2,11,2                     []
             2,1,11          [1, 164, 244]
             2,1,1                   [847]
             2,1,2                      []
             2,2,11                 [1055]
             2,2,1                   [359]
             2,2,2                   [247]
1,1,2        11,11,1                    []

In [86]:
# Group edges_df by 'select_index' and aggregate all res_id lists from nearby indices
def flatten(lists):
    return [item for sublist in lists for item in sublist]

nearby_resid_df = (
    edges_df.groupby('select_index')['res_id']
    .apply(lambda x: flatten([v if isinstance(v, (list, np.ndarray)) else [] for v in x]))
    .reset_index()
    .rename(columns={'res_id': 'nearby_res_id'})
)

nearby_resid_df

,select_index,nearby_res_id
0,"1,1,1","[1181, 147, 127, 780, 704, 1366, 274, 660, 226..."
1,"1,1,10","[1442, 1013, 1181, 195, 1322, 127, 780, 190, 4..."
2,"1,1,11","[1013, 1181, 195, 1322, 127, 780, 704, 274, 66..."
3,"1,1,2","[147, 619, 1474, 704, 1366, 1057, 226, 1031, 8..."
4,"1,1,3","[147, 619, 1474, 542, 1299, 1366, 1057, 897, 1..."
...,...,...
1326,"9,9,5","[1178, 820, 719, 836, 275, 467, 1205, 995, 710..."
1327,"9,9,6","[820, 571, 1405, 836, 275, 467, 1205, 710, 461..."
1328,"9,9,7","[820, 571, 1405, 923, 275, 467, 1205, 172, 814..."
1329,"9,9,8","[571, 1405, 923, 403, 172, 814, 1266, 711, 845..."


In [ ]:
#CHANGES: ADDED MODULES
nearby_resid_df = flatten_subboxes(edges_df)
nearby_resid_df

,select_index,nearby_res_id
0,"1,1,1","[1181, 147, 127, 780, 704, 1366, 274, 660, 226..."
1,"1,1,10","[1442, 1013, 1181, 195, 1322, 127, 780, 190, 4..."
2,"1,1,11","[1013, 1181, 195, 1322, 127, 780, 704, 274, 66..."
3,"1,1,2","[147, 619, 1474, 704, 1366, 1057, 226, 1031, 8..."
4,"1,1,3","[147, 619, 1474, 542, 1299, 1366, 1057, 897, 1..."
...,...,...
1326,"9,9,5","[1178, 820, 719, 836, 275, 467, 1205, 995, 710..."
1327,"9,9,6","[820, 571, 1405, 836, 275, 467, 1205, 710, 461..."
1328,"9,9,7","[820, 571, 1405, 923, 275, 467, 1205, 172, 814..."
1329,"9,9,8","[571, 1405, 923, 403, 172, 814, 1266, 711, 845..."


## Modeling molecules as spheres

In [ ]:
#rewrite these without pd.df and python lists
def mic_vector(dx, box_length):
    """Minimum image displacement for array dx."""
    return dx - np.rint(dx / box_length) * box_length

def mic_distance(p1, p2, box_length):
    """Minimum image Euclidean distance."""
    dx = mic_vector(p2 - p1, box_length)
    return np.linalg.norm(dx)

def mec_pbc_miniball(positions, box_length, references=None):
    """
    Finds approximate minimum enclosing sphere under PBC using the miniball library.
    positions: (n,3) array
    box_length: float
    references: list of indices to try as reference points (default: all points)
    Returns: center (3,), radius (float)
    """
    n = len(positions)
    if references is None:
        references = range(n)
    
    best_center = None
    best_radius = np.inf
    
    for i in references:
        ref = positions[i]
        
        # Unwrap positions relative to reference
        diffs = positions - ref
        diffs = mic_vector(diffs, box_length)
        unwrapped = ref + diffs
        
        # Compute Euclidean MEB via miniball
        center_unwrapped, radius_unwrapped_sq = miniball.get_bounding_ball(unwrapped)
        radius_unwrapped = np.sqrt(radius_unwrapped_sq)
        
        # Map center into primary box
        center_primary = np.mod(center_unwrapped, box_length)
        
        # Check radius under true PBC
        dists = np.array([mic_distance(center_primary, p, box_length) for p in positions])
        radius_check = np.max(dists)
        
        if radius_check < best_radius:
            best_radius = radius_check
            best_center = center_primary
    
    return best_center, best_radius

In [ ]:
def compute_centers_radii(dataframe, box_length):
    """
    Given a multi-indexed DataFrame (dataframe) and box_length,
    returns a DataFrame with columns ['res_id', 'center_x', 'center_y', 'center_z', 'radius']
    containing the minimum enclosing sphere center and radius for each residue.
    """
    results = []
    for res_id in dataframe.index.get_level_values(0).unique():
        atoms = dataframe.loc[res_id]
        positions = atoms[['x', 'y', 'z']].values.astype(float)
        center, radius = mec_pbc_miniball(positions, box_length)
        results.append({'res_id': res_id, 'center': center, 'radius': radius})

    centers_radii_df = pd.DataFrame([{
        'res_id': r['res_id'],
        'center_x': r['center'][0],
        'center_y': r['center'][1],
        'center_z': r['center'][2],
        'radius': r['radius']
    } for r in results])

    return centers_radii_df

In [95]:
centers_radii_df = compute_centers_radii(data, box_length)
print(centers_radii_df)

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1mUntyped global name 'mec_pbc_miniball':[0m [1m[1mCannot determine Numba type of <class 'function'>[0m
[1m
File "..\..\..\..\..\..\..\..\AppData\Local\Temp\ipykernel_8448\2198273877.py", line 12:[0m
[1m<source missing, REPL/exec in use?>[0m
[0m
[0m[1mDuring: Pass nopython_type_inference[0m 

This error may have been caused by the following argument(s):
- argument 0: [1mCannot determine Numba type of <class 'pandas.core.frame.DataFrame'>[0m


## Blocking algorithm

In [ ]:
from numpy.linalg import norm

# A function to check if a direct path between two spheres is blocked by any third sphere

def is_blocked(center1, center2, centers_radii_df, tol=1e-8):
    """
    Test if any third sphere blocks the direct path between two spheres. Returns True if blocked, False otherwise.

    centers_radii_df: DataFrame with columns ['center_x', 'center_y', 'center_z', 'radius']
    tol: float, numerical tolerance
    """

    # Vector from center1 to center2
    p1 = np.array(center1)
    p2 = np.array(center2)
    d = p2 - p1
    d_norm = norm(d)
    if d_norm < tol:
        return False  # same center

    # For each third sphere
    for idx, row in centers_radii_df.iterrows():
        c = np.array([row['center_x'], row['center_y'], row['center_z']])
        r = row['radius']

        # Skip if c is center1 or center2
        if np.allclose(c, p1, atol=tol) or np.allclose(c, p2, atol=tol):
            continue

        # Project c onto the line segment p1-p2
        t = np.dot(c - p1, d) / (d_norm ** 2)
        t = np.clip(t, 0, 1)
        closest = p1 + t * d
        dist = norm(c - closest)

        if dist < r - tol:
            return True  # blocked

    return False  # not blocked

In [ ]:
def get_not_blocked_pairs(centers_radii_df):
    """
    Returns a list of tuples (res_id1, res_id2) for residue pairs that are NOT blocked.
    """
    not_blocked_pairs = []

    for i in range(len(centers_radii_df)):
        for j in range(i + 1, len(centers_radii_df)):
            c1 = centers_radii_df.iloc[i][['center_x', 'center_y', 'center_z']]
            c2 = centers_radii_df.iloc[j][['center_x', 'center_y', 'center_z']]
            blocked = is_blocked(c1, c2, centers_radii_df)
            if not blocked:
                res_id1 = int(centers_radii_df.iloc[i]['res_id'])
                res_id2 = int(centers_radii_df.iloc[j]['res_id'])
                not_blocked_pairs.append((res_id1, res_id2))
                
    return not_blocked_pairs

In [ ]:
# For each row in nearby_resid_df, get the centers/radii for the nearby residues,
# apply get_not_blocked_pairs, and store the result in a list.

blocked_pairs_by_subbox = []

for idx, row in nearby_resid_df.iterrows():
    nearby_ids = row['nearby_res_id']
    # Filter centers_radii_df for these residues
    sub_centers = centers_radii_df[centers_radii_df['res_id'].isin(nearby_ids)]
    # Only check if there are at least 2 residues
    if len(sub_centers) < 2:
        blocked_pairs_by_subbox.append([])
        continue
    # Get not blocked pairs for this subbox
    pairs = get_not_blocked_pairs(sub_centers)
    blocked_pairs_by_subbox.append(pairs)

# blocked_pairs_by_subbox[i] contains the not-blocked pairs for nearby_resid_df.iloc[i]
blocked_pairs_df = pd.DataFrame({
    'select_index': nearby_resid_df['select_index'],
    'not_blocked_pairs': blocked_pairs_by_subbox
})
print(blocked_pairs_df)

This code takes 7m 31.8s to execute

In [ ]:
# Flatten all not_blocked_pairs and collect unique pairs as sorted tuples
unique_pairs = set()
for pairs in blocked_pairs_df['not_blocked_pairs']:
    for pair in pairs:
        # Ensure (a, b) and (b, a) are treated as the same
        unique_pairs.add(tuple(sorted(pair)))

unique_pairs_list = list(unique_pairs)
print(f"Number of connection: {len(unique_pairs_list)}")

# Example: print first 10 pairs
print(unique_pairs_list[:10])

### Visualising the 3D-graph

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot all molecule centers as scatter points
ax.scatter(
    centers_radii_df['center_x'],
    centers_radii_df['center_y'],
    centers_radii_df['center_z'],
    c='lightgreen', s=10, alpha=0.6, label='Molecule Centers'
)

# Draw edges for each pair in unique_pairs_list
for res_id1, res_id2 in unique_pairs_list:
    p1 = centers_radii_df.loc[centers_radii_df['res_id'] == res_id1, ['center_x', 'center_y', 'center_z']].values
    p2 = centers_radii_df.loc[centers_radii_df['res_id'] == res_id2, ['center_x', 'center_y', 'center_z']].values
    if len(p1) > 0 and len(p2) > 0:
        xs = [p1[0][0], p2[0][0]]
        ys = [p1[0][1], p2[0][1]]
        zs = [p1[0][2], p2[0][2]]
        ax.plot(xs, ys, zs, c='gray', alpha=0.3, linewidth=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Molecular Connections')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
G = nx.Graph()

G.add_edges_from(unique_pairs_list)

print(G)

# Trying out Gabriel Graph

In [ ]:
"""
Gabriel Graph

Has significantly less edges than the edges algorithm we did. This is because they find different things, for gabriel graphs give the nearest neighbours 
by defining that no other vertex can exist between a region between two vertices. We define it as all vertices within a region of 3x3 subbox are considered
near to each other.

Gabriel Graph however does not give us which points are closest to a given edge, making it so that any blocking algorithm we make would have to iterate over all nodes
within the system. The subbox method on the other hand tells us which nodes are in the vicinity of a given edge, therefore minimizing computation time.

"""

import itertools
import networkx as nx
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay

# Trying Gabriel Graph
def DelaunayGraph(points):
    tri = Delaunay(points)
    G = nx.Graph()

    indptr = tri.vertex_neighbor_vertices[0]
    indices = tri.vertex_neighbor_vertices[1]
    for i in range(len(points)):
        for j in indices[indptr[i]:indptr[i+1]]:
            if i < j:
                G.add_edge(i,j)

    return G

points = centre[['x','y','z']].values
gabrielGraph = DelaunayGraph(points)

for e in gabrielGraph.edges():
    for other in gabrielGraph.nodes():
        if other not in e:
            p1 = points[e[0]]
            p2 = points[e[1]]
            p3 = points[other]
            center = 0.5 * (p1 + p2)
            radius = 0.5 * np.linalg.norm(p2-p1)
            if (np.linalg.norm(p3-center) <= radius):
                gabrielGraph.remove_edge(e[0],e[1])
                break



In [ ]:
# #Making edges
# edges = []
# new_edges = []

# ## Step 1: Connect edges nodes within subbox
# def generate_combinations(items, x):
#     result = []
    
#     def backtrack(start, current):
#         if len(current) == x:
#             result.append(tuple(current))
#             return
#         for i in range(start, len(items)):
#             current.append(items[i])
#             backtrack(i + 1, current)
#             current.pop()
    
#     backtrack(0, [])
#     return result

# for n in subbox.index:
#     selected_box = [list(subbox.at[n,"index"])]
#     nodes = subbox.at[n,"res_id"]
#     if len(nodes) > 1:
#         combin = generate_combinations(nodes,2)
#         for (x,y) in combin:
#             # if ((x,y) not in edges or (y,x) not in edges):
#             edges.append((x,y))
# ## Step 2: Find adjacent boxes
#     adjacent_boxes = []
#     for (p,q,r) in selected_box:
#         for i in [-1,0,1]:
#             for j in [-1,0,1]:
#                 for k in [-1,0,1]:
#                     adjacent_boxes.append((p+i,q+j,r+k))
#         adjacent_boxes.remove((p,q,r))
# ## Step 3: Connect nodes in selected subbox to nodes in adjacent subbox 
#         for (p,q,r) in adjacent_boxes:
#             try: 
#                 adjacent_nodes = subbox.loc[subbox["index"] == (p, q, r), "res_id"]
#                 if adjacent_nodes.empty:
#                     continue
#                 adjacent_nodes = adjacent_nodes.values[0]
#             except:
#                 print(f"fail at {(p,q,r)}")

#             for x in adjacent_nodes:
#                 for j in nodes:
#                     if ([j,x] not in new_edges or [x,j] not in new_edges):
#                         new_edges.append((j, x)) 


# edges = [tuple(map(int,group)) for group in edges]
# new_edges = [tuple(map(int,group)) for group in new_edges]

# total_edges = edges + new_edges
# len(total_edges)

In [ ]:
# #Uncomment the following to visualize the edges
# import networkx as nx
# import matplotlib.pyplot as plt

# # Create graph from all edges
# G = nx.Graph(edges)
# H = nx.Graph(new_edges)
# H.add_edges_from(edges)

# # Plot the graphs
# plt.figure(figsize=(12, 8))
# plt.subplot(121)   

# plt.title("Edges within Subbox")
# plt.subplot(122)
# nx.draw(H, with_labels=False, node_size=1, edge_color='red', width=0.1)
# plt.title("Edges between Subboxes")

In [ ]:
# ## Step 4: assign weightage to each edge
# total_edges = edges + new_edges # Can change this later to only have "edges", I made new_edges to show the step by step progression of the number of edges

# #blocking code

# for (p,q) in total_edges: 
#     molecule_1 = centre.loc[centre["res_id"] == p, ['x','y','z']].values[0]
#     molecule_2 = centre.loc[centre["res_id"] == q, ['x','y','z']].values[0]
#     distance = float(np.sum(np.square(np.add(molecule_1,molecule_2))) ** (1/2))
#     total_edges[total_edges.index((p,q))] = (p,q, distance)

# len(total_edges)

In [ ]:
# import networkx as nx
# Weighted_graph = nx.Graph()
# Weighted_graph.add_weighted_edges_from(total_edges)
# nx.draw(Weighted_graph, with_labels=False, node_size=1, edge_color='red', width=0.1)
# plt.title("Graph with weighted edges")